# T4 + T5 — chỉ mục dense, truy hồi 3 nhánh, gộp RRF (Kaggle, 1 tài khoản)

**Cài đặt:** Accelerator = *GPU T4* (chỉ để mã hoá 1.200 câu hỏi; phần còn lại chạy CPU), Internet = *On*.
**Đầu vào:** `mir-code`, `mir-data`, `mir-bm25`, và kết quả T3 của 3 người: `mir-emb-m0`, `mir-emb-m1`, `mir-emb-m2`
(mỗi dataset chứa `emb/structure/shard-XXXX/`).
**Đầu ra:** `/kaggle/working/runs/` (run từng nhánh + 7 cấu hình A–G + gộp có trọng số) và `index/structure/dense.faiss`.
Bộ nhớ: IVF-SQ8 cho ~8,6 triệu × 1024 chiều ≈ 9 GB (RAM ~30 GB đủ); LSR đọc thẳng từng shard emb (không cần chép).

In [ ]:
# Chuẩn bị chung: chép code, đặt biến môi trường, nối (symlink) dữ liệu đầu vào vào thư mục làm việc
import os, shutil, subprocess, sys, json, glob
CODE_IN = '/kaggle/input/mir-code'          # dataset: mir/, configs/
DATA_IN = '/kaggle/input/mir-data'          # dataset: Data/ (query.parquet, links_corpus.parquet), corpus_clean/, chunks/
WORK = '/kaggle/working'
CODE = f'{WORK}/code'
if not os.path.exists(CODE):
    shutil.copytree(CODE_IN, CODE)
os.chdir(CODE)
os.environ.update(MIR_DATA_DIR=DATA_IN, MIR_WORK_DIR=WORK, HF_HUB_DISABLE_SYMLINKS_WARNING='1')
for name in ('corpus_clean', 'chunks'):
    if not os.path.exists(f'{WORK}/{name}') and os.path.exists(f'{DATA_IN}/{name}'):
        os.symlink(f'{DATA_IN}/{name}', f'{WORK}/{name}')
def sh(cmd):
    print('$', cmd); r = subprocess.run(cmd, shell=True); assert r.returncode == 0, f'lỗi: {cmd}'
sh('pip install -q pyyaml pyvi jieba faiss-cpu')
print(subprocess.run(f'df -h {WORK}; free -g; nproc', shell=True, capture_output=True, text=True).stdout)
CFG = 'configs/baseline.yaml'

In [ ]:
EMB = ','.join(sorted(glob.glob('/kaggle/input/mir-emb-m*/emb/structure')))
print('emb:', EMB)
os.makedirs(f'{WORK}/index/structure', exist_ok=True)
if not os.path.exists(f'{WORK}/index/structure/bm25'):
    os.symlink('/kaggle/input/mir-bm25/index/structure/bm25', f'{WORK}/index/structure/bm25')
# Mã hoá câu hỏi (dense + sparse) — dùng chung cho nhánh dense, LSR và cho đo nlist/nprobe
sh(f'python -m mir.retrieve qencode --config {CFG} --device cuda:0')

In [ ]:
# Chọn nlist/nprobe: Recall@100 của IVF-SQ8 so với IndexFlatIP trên mẫu ~100k đoạn, truy vấn = câu hỏi thật
sh(f'python -m mir.index dense-tune --config {CFG} --emb {EMB} --qemb {WORK}/qemb/query.npz')
# → xem bảng, sửa index.dense.nlist / nprobe trong configs/baseline.yaml (bản trong {CODE}) nếu cần, rồi chạy tiếp

In [ ]:
# Dựng chỉ mục dense (IVF-SQ8) cho toàn kho
sh(f'python -m mir.index dense --config {CFG} --emb {EMB}')
print(open(f'{WORK}/index/structure/dense.json').read())

In [ ]:
# Truy hồi 3 nhánh (độ sâu 1000) rồi gộp: 7 cấu hình RRF A–G + gộp có trọng số
sh(f'python -m mir.retrieve run --config {CFG} --branch all --emb {EMB}')
sh(f'python -m mir.fuse --config {CFG}')
sh(f'ls -la {WORK}/runs; du -sh {WORK}/index')